<a href="https://colab.research.google.com/github/manas150309-debug/3cse5_manas_gupta_practiclefile/blob/main/houseprediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CUSTOMER SEGMENTATION USING AUTOENCODER + KMEANS
# FULL GOOGLE COLAB CODE
# ============================================================

# -------------------------------
# 1. Upload kaggle.json
# -------------------------------
from google.colab import files
uploaded = files.upload()   # Upload your kaggle.json file here

# -------------------------------
# 2. Setup Kaggle API
# -------------------------------
import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)

if os.path.exists('kaggle.json'):
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
else:
    raise FileNotFoundError("kaggle.json not found. Please upload it again.")

# -------------------------------
# 3. Install required libraries
# -------------------------------
!pip install -q kaggle tensorflow scikit-learn pandas matplotlib seaborn

# -------------------------------
# 4. Download dataset from Kaggle
# -------------------------------
!kaggle datasets download -d vjchoudhary7/customer-segmentation-tutorial-in-python

# -------------------------------
# 5. Unzip dataset
# -------------------------------
!unzip -o customer-segmentation-tutorial-in-python.zip

# -------------------------------
# 6. Import libraries
# -------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# -------------------------------
# 7. Load dataset
# -------------------------------
df = pd.read_csv('/content/Mall_Customers.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

# -------------------------------
# 8. Data preprocessing
# -------------------------------
# Convert Gender to numeric
label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])   # Female/Male -> numeric

# Choose features
features = ['Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nScaled data shape:", X_scaled.shape)

# -------------------------------
# 9. Build Autoencoder
# -------------------------------
input_dim = X_scaled.shape[1]
encoding_dim = 2   # latent dimension

input_layer = Input(shape=(input_dim,))

# Encoder
x = Dense(8, activation='relu')(input_layer)
x = Dense(4, activation='relu')(x)
latent = Dense(encoding_dim, activation='linear', name='latent_space')(x)

# Decoder
x = Dense(4, activation='relu')(latent)
x = Dense(8, activation='relu')(x)
output_layer = Dense(input_dim, activation='linear')(x)

# Create models
autoencoder = Model(inputs=input_layer, outputs=output_layer)
encoder = Model(inputs=input_layer, outputs=latent)

# Compile model
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nAutoencoder Summary:")
autoencoder.summary()

# -------------------------------
# 10. Train Autoencoder
# -------------------------------
history = autoencoder.fit(
    X_scaled,
    X_scaled,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# -------------------------------
# 11. Extract encoded features
# -------------------------------
encoded_data = encoder.predict(X_scaled)

print("\nEncoded feature shape:", encoded_data.shape)

# -------------------------------
# 12. Find best number of clusters
# -------------------------------
silhouette_scores = []
cluster_range = range(2, 8)

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(encoded_data)
    score = silhouette_score(encoded_data, labels)
    silhouette_scores.append(score)
    print(f"Clusters = {k}, Silhouette Score = {score:.4f}")

best_k = cluster_range[np.argmax(silhouette_scores)]
print("\nBest number of clusters:", best_k)

# -------------------------------
# 13. Final KMeans clustering
# -------------------------------
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(encoded_data)

df['Cluster'] = clusters

final_score = silhouette_score(encoded_data, clusters)
print("\nFinal Silhouette Score:", round(final_score, 4))

# -------------------------------
# 14. Plot training loss
# -------------------------------
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Autoencoder Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 15. Plot silhouette scores
# -------------------------------
plt.figure(figsize=(8, 5))
plt.plot(list(cluster_range), silhouette_scores, marker='o')
plt.title('Silhouette Score for Different Cluster Counts')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# -------------------------------
# 16. Visualize clusters in latent space
# -------------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=encoded_data[:, 0],
    y=encoded_data[:, 1],
    hue=clusters,
    palette='Set2',
    s=100
)
plt.title('Customer Segmentation using Autoencoder + KMeans')
plt.xlabel('Latent Feature 1')
plt.ylabel('Latent Feature 2')
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

# -------------------------------
# 17. Visualize clusters using original features
# -------------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Cluster',
    palette='Set2',
    s=100
)
plt.title('Clusters by Income vs Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.grid(True)
plt.show()

# -------------------------------
# 18. Cluster summary
# -------------------------------
print("\nCluster-wise Mean Values:")
cluster_summary = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()
print(cluster_summary)

print("\nCustomers in each cluster:")
print(df['Cluster'].value_counts().sort_index())

# -------------------------------
# 19. Save output file
# -------------------------------
output_file = '/content/customer_segmented_output.csv'
df.to_csv(output_file, index=False)
print(f"\nClustered dataset saved as: {output_file}")

# -------------------------------
# 20. Download output file
# -------------------------------
files.download(output_file)
